# Model Training — Predictive Maintenance

Trains **LightGBM** and **RandomForest** classifiers on a pandas feature frame.
Target: `had_failure`.

Both models are logged with MLflow and **registered in the workspace model
registry** (`maintenance_lgbm`, `maintenance_rf`), so each appears as an ML model
item alongside the experiment. Training uses named, primitive-typed pandas
columns and an explicit `infer_signature` with a named output column — the
**column-based signature** required to publish a model version as a REST endpoint.

**Reads:** `gold_ml_features`  **Writes:** `gold_ml_model_metrics`, `gold_ml_scoring_meta`, MLflow registry

In [ ]:
import mlflow
import mlflow.lightgbm
import mlflow.sklearn
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from mlflow.models.signature import infer_signature
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp, when, isnan

spark = SparkSession.builder.getOrCreate()

EXPERIMENT_NAME = 'manufacturing-qc-predictive-maintenance'
mlflow.set_experiment(EXPERIMENT_NAME)
print(f'Spark session ready — MLflow experiment: {EXPERIMENT_NAME}')

In [ ]:
features_df = spark.read.table('gold_ml_features')
print(f'Feature rows: {features_df.count():,}')

# Clean nulls / NaN in numeric columns so the trainer never sees invalid values
for c, dtype in features_df.dtypes:
    if dtype in ('double', 'float'):
        features_df = features_df.withColumn(c, when(col(c).isNull() | isnan(col(c)), lit(0.0)).otherwise(col(c)))
    elif dtype in ('int', 'bigint', 'long'):
        features_df = features_df.withColumn(c, when(col(c).isNull(), lit(0)).otherwise(col(c)))

features_df.groupBy('had_failure').count().show()

In [ ]:
# === Build a pandas feature frame with NAMED, PRIMITIVE-TYPED columns ===
# No VectorAssembler: a vector column gives the logged model a tensor schema,
# and Fabric cannot publish tensor-schema models as REST endpoints
# ("This model's schema is either empty or has tensors").
LABEL = 'had_failure'

# Auto-discover numeric feature columns (exclude label and leakage columns)
exclude = ["had_failure", "failure_event_count", "total_failure_downtime", "total_failure_defects", "had_failure"]
numeric_features = [
    c for c, dtype in features_df.dtypes
    if dtype in ('double', 'float', 'int', 'bigint', 'long') and c not in exclude
]
cat_cols = []
category_maps = {}

pdf = features_df.select(numeric_features + [LABEL]).toPandas()
for c in numeric_features:
    pdf[c] = pd.to_numeric(pdf[c], errors='coerce').fillna(0.0)

feature_cols = list(numeric_features)

X = pdf[feature_cols].astype('float64')
y = pdf[LABEL].astype('int64')

print(f'Model-ready rows: {len(X):,}')
print(f'Feature count: {len(feature_cols)}')

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train):,} rows')
print(f'Test:  {len(X_val):,} rows')

In [ ]:
# lgbm_model
# Autolog captures params/metrics/plots; the model itself is logged EXPLICITLY
# with infer_signature + a named pandas output column so both input and output
# schemas are column-based (endpoint-publishable).
mlflow.lightgbm.autolog(log_models=False)
mlflow.sklearn.autolog(log_models=False)

lgbm_model = LGBMClassifier(
    learning_rate=0.07,
    max_delta_step=2,
    n_estimators=100,
    max_depth=10,
    objective='binary',
    random_state=42,
)

with mlflow.start_run(run_name='maintenance_lgbm') as run:
    lgbm_run_id = run.info.run_id  # Capture run_id for model prediction later
    lgbm_model.fit(X_train, y_train)
    y_pred = lgbm_model.predict(X_val)
    proba = lgbm_model.predict_proba(X_val)[:, 1]
    auc_lgbm = float(roc_auc_score(y_val, proba))
    acc_lgbm = float(accuracy_score(y_val, y_pred))
    f1_lgbm = float(f1_score(y_val, y_pred))

    signature = infer_signature(X_val, pd.DataFrame({'prediction': y_pred.astype('int64')}))
    mlflow.lightgbm.log_model(
        lgbm_model,
        'model',
        signature=signature,
        input_example=X_val.head(5),
        registered_model_name='maintenance_lgbm',  # Register the trained model
    )

print(f'LightGBM run: {lgbm_run_id}')
print(f'  AUC {auc_lgbm:.4f} | Accuracy {acc_lgbm:.4f} | F1 {f1_lgbm:.4f}')

In [ ]:
# rf_model — second registered model, mirroring the multi-model workspace layout
rf_model = RandomForestClassifier(
    n_estimators=60,
    max_depth=8,
    random_state=42,
    n_jobs=-1,
)

with mlflow.start_run(run_name='maintenance_rf') as run:
    rf_run_id = run.info.run_id
    rf_model.fit(X_train, y_train)
    y_pred_rf = rf_model.predict(X_val)
    proba_rf = rf_model.predict_proba(X_val)[:, 1]
    auc_rf = float(roc_auc_score(y_val, proba_rf))
    acc_rf = float(accuracy_score(y_val, y_pred_rf))
    f1_rf = float(f1_score(y_val, y_pred_rf))

    signature = infer_signature(X_val, pd.DataFrame({'prediction': y_pred_rf.astype('int64')}))
    mlflow.sklearn.log_model(
        rf_model,
        'model',
        signature=signature,
        input_example=X_val.head(5),
        registered_model_name='maintenance_rf',
    )

print(f'RandomForest run: {rf_run_id}')
print(f'  AUC {auc_rf:.4f} | Accuracy {acc_rf:.4f} | F1 {f1_rf:.4f}')

# Champion = better validation AUC; scoring loads this from the registry
if auc_lgbm >= auc_rf:
    champion_name, champion_flavor, champion_run = 'maintenance_lgbm', 'lightgbm', lgbm_run_id
    auc, accuracy, f1 = auc_lgbm, acc_lgbm, f1_lgbm
else:
    champion_name, champion_flavor, champion_run = 'maintenance_rf', 'sklearn', rf_run_id
    auc, accuracy, f1 = auc_rf, acc_rf, f1_rf

print(f'\n=== Model Evaluation ===')
print(f'Champion: {champion_name}')
print(f'AUC-ROC:  {auc:.4f}')
print(f'Accuracy: {accuracy:.4f}')
print(f'F1:       {f1:.4f}')

In [ ]:
metrics_df = spark.createDataFrame(
    [('manufacturing-qc', 'predictive-maintenance', champion_name,
      len(feature_cols), len(X_train), len(X_val),
      float(auc), float(accuracy), float(f1))],
    ['demo_id', 'use_case', 'model_type', 'feature_count',
     'train_rows', 'test_rows', 'auc_roc', 'accuracy', 'f1_score']
).withColumn('trained_at', current_timestamp())

metrics_df.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_model_metrics')
print('Model metrics saved to gold_ml_model_metrics')
metrics_df.show(truncate=False)

In [ ]:
# Persist scoring metadata: champion model + flavor, feature order, category maps.
# Downstream notebooks load the model from the MLflow REGISTRY (not Files/).
import json

scoring_meta = {
    'champion_model': champion_name,
    'champion_flavor': champion_flavor,
    'champion_run_id': champion_run,
    'feature_cols': feature_cols,
    'numeric_features': numeric_features,
    'cat_cols': cat_cols,
    'category_maps': category_maps,
    'label': 'had_failure',
}
meta_df = spark.createDataFrame([(json.dumps(scoring_meta),)], ['meta_json'])
meta_df.write.mode('overwrite').format('delta').saveAsTable('gold_ml_scoring_meta')

print('Registered models: maintenance_lgbm, maintenance_rf')
print(f'Champion for scoring: {champion_name}')
print('Scoring metadata saved to gold_ml_scoring_meta')